In [ ]:
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from torch import nn
import numpy as np


In [ ]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.version.cuda)
print(torch.cuda.get_arch_list())

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
window_size = 30
pad_value = 0

In [ ]:
exclude_rgb_vis=False
exclude_weather_features=True
exclude_canopy_temp=True
dataset_name = "stratified_train_test_datasets_v3_all_interpolated.xlsx"

In [ ]:
import os
import random

seed = 46
def reset_seed_all(_seed):
    os.environ["PYTHONHASHSEED"] = str(_seed)
    random.seed(_seed)

    torch.manual_seed(_seed)
    np.random.seed(_seed)

    torch.cuda.manual_seed_all(_seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)

    g = torch.Generator()
    g.manual_seed(_seed)
    return g

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

In [ ]:
from features import *

vi_features = get_vi_features(exclude_rgb_vis=exclude_rgb_vis)
weather_features = get_weather_features(exclude=exclude_weather_features)
canopy_temp_features = get_canopy_temp_features(exclude=exclude_canopy_temp)
features = vi_features + weather_features + canopy_temp_features

output_variable = get_output_variable()

In [ ]:
# Loading saved training test
train_df = pd.read_excel(f"data/{dataset_name}", sheet_name='Train')
test_df = pd.read_excel(f"data/{dataset_name}", sheet_name='Test')

sns.kdeplot(train_df[output_variable], label='Train')
sns.kdeplot(test_df[output_variable], label='Test')
plt.legend()
plt.title("Train vs Test Yield Distribution")

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler

scaler_X = StandardScaler()
scaler_y = RobustScaler()

# Fit on training features and yield only
scaler_X.fit(train_df[features])
scaler_y.fit(train_df[[output_variable]])

train_df_scaled = train_df.copy()
test_df_scaled  = test_df.copy()

# --- Scaled features and target variable---
train_df_scaled[features] = scaler_X.transform(train_df[features])
train_df_scaled[output_variable] = scaler_y.transform(train_df[[output_variable]])

test_df_scaled[features] = scaler_X.transform(test_df[features])
test_df_scaled[output_variable] = scaler_y.transform(test_df[[output_variable]])

In [ ]:

num_na_rows = train_df_scaled.isna().any(axis=1).sum()
print(f"train_df_scaled Rows with at least one NaN: {num_na_rows}")

num_na_rows = test_df_scaled.isna().any(axis=1).sum()
print(f"test_df_scaled Rows with at least one NaN: {num_na_rows}")


In [ ]:

def train_data(model, model_train_loader, num_epochs, learning_rate, weight_decay, hyper_param_criterion_method, file_save_name, stop_early = False):
    best_train_loss  = float('inf')
    patience = 20
    counter = 0

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )
    if hyper_param_criterion_method == "MSE":
        criterion = nn.MSELoss()
    else:
        criterion = nn.SmoothL1Loss(beta=1)
        # criterion = nn.SmoothL1Loss(beta=1.0, reduction="none")

    T = 46  # e.g., 45
    total_loss = []
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0
        for X_batch, y_batch, lengths in model_train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            lengths = lengths.to(device)
            optimizer.zero_grad()
            y_pred = model(X_batch, lengths)

            # loss_per_sample = criterion(y_pred, y_batch)
            loss = criterion(y_pred, y_batch)

            # weights = lengths.float() / T                 # shape: (batch,)
            # weights = weights / weights.mean()
            # loss = (weights * loss_per_sample).mean()

            loss.backward()
            # Gradient clipping (prevents exploding gradients)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            optimizer.step()
            train_loss += loss.item() * X_batch.size(0)

        avg_loss = train_loss / len(model_train_loader.dataset)
        total_loss.append(avg_loss)

        # print(f"Epoch [{epoch+1}/{num_epochs}] - Train Loss: {avg_loss:.5f}")

        if train_loss < best_train_loss:
            best_train_loss = train_loss
            if file_save_name != "":
                torch.save(model.state_dict(), file_save_name)
            counter = 0
        elif stop_early:
            counter += 1
            if counter >= patience:
                print("Early stopping triggered.")
                break
    return total_loss


In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


def evaluate_model(model, val_loader, output_variable_scaler, do_inverse_transform = False, plot_pred_vs_true = False):

    model.eval()  # set model to evaluation mode
    y_true, y_pred = [], []

    with torch.no_grad():
        for X_batch, y_batch, lengths in val_loader:
            X_batch = X_batch.to(device)
            lengths = lengths.to(device)
            y_hat = model(X_batch, lengths)

            y_true.append(y_batch.cpu())
            y_pred.append(y_hat.cpu())

    y_true = torch.cat(y_true).numpy()
    y_pred = torch.cat(y_pred).numpy()

    if do_inverse_transform:
        y_true = output_variable_scaler.inverse_transform(y_true)
        y_pred = output_variable_scaler.inverse_transform(y_pred)

    r2 = r2_score(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)

    if plot_pred_vs_true:
        plt.figure(figsize=(8,6))
        plt.scatter(y_true, y_pred, alpha=0.7)
        plt.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'k--')
        plt.xlabel("Actual Yield")
        plt.ylabel("Predicted Yield")
        plt.title("GRU Predictions vs Actual")
        plt.show()

    return r2, mse, mae

In [ ]:
def predict_yield(model, val_loader, output_variable_scaler, do_inverse_transform = False, plot_pred_vs_true = False):

    model.eval()  # set model to evaluation mode
    y_true, y_pred = [], []

    with torch.no_grad():
        for X_batch, y_batch, lengths in val_loader:
            X_batch = X_batch.to(device)
            lengths = lengths.to(device)
            y_hat = model(X_batch, lengths)

            y_true.append(y_batch.cpu())
            y_pred.append(y_hat.cpu())

    y_true = torch.cat(y_true).numpy()
    y_pred = torch.cat(y_pred).numpy()

    if do_inverse_transform:
        y_true = output_variable_scaler.inverse_transform(y_true)
        y_pred = output_variable_scaler.inverse_transform(y_pred)

    return y_true, y_pred

In [ ]:
from torch.nn.utils.rnn import pack_padded_sequence
from helper import make_progressive_windows
from vi_dataset import VIDataset
from model_definitions import  GRUModel
from torch.utils.data import DataLoader

def validation_with_test(trial_id, rand_seed, batch_size, epochs, hidden_size, num_layers, dropout, bidirectional, learning_rate, weight_decay, hyper_param_criterion_method, early_stopping = True):

    torch_gen = reset_seed_all(rand_seed)

    X_train_scaled, y_train_scaled, lengths_train, no_days_train  = make_progressive_windows(
            dataframe=train_df_scaled,
            features=features,
            output_variable=output_variable,
            window_size=window_size,
            pad_value=pad_value)

    X_test_scaled, y_test_scaled, lengths_test, no_days_test = make_progressive_windows(
        dataframe=test_df_scaled,
        features=features,
        output_variable=output_variable,
        window_size=window_size,
        pad_value=pad_value)

    train_dataset = VIDataset(X_train_scaled, y_train_scaled, lengths_train)
    test_dataset = VIDataset(X_test_scaled, y_test_scaled, lengths_test)

    train_loader  = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers =0, worker_init_fn=seed_worker, generator=torch_gen)
    test_loader    = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers =0, worker_init_fn=seed_worker, generator=torch_gen)

    model = GRUModel(input_size = len(features), hidden_size = hidden_size, num_layers = num_layers,
                     output_size= 1 , dropout=dropout, bidirectional = bidirectional).to(device)

    train_data(
        model = model,
        model_train_loader= train_loader,
        num_epochs = epochs,
        learning_rate = learning_rate,
        weight_decay = weight_decay,
        hyper_param_criterion_method = hyper_param_criterion_method,
        file_save_name=f"best_model_hyper_param{trial_id}.pth",
        stop_early=early_stopping
    )

    r2, mse, mae = evaluate_model(
        model = model,
        val_loader = test_loader,
        output_variable_scaler=scaler_y)


    print(f'Test R²: {r2:.4f}, Test MSE: {mse:.4f}, Test MAE: {mae:.4f}')


    return r2

In [ ]:
from helper import filter_window_data_by_length

total_n_days = 46 # (from may 1st to June 15th)

def get_results_by_day():
    all_test_r2 = []
    all_test_mse = []
    all_test_mae = []
    for day in range(1, total_n_days + 1):
        # print(f"day: {day}")

        x_train_scaled_filtered, y_train_scaled_filtered, lengths_train_filtered = filter_window_data_by_length(
            X_train_scaled, y_train_scaled, valid_length_train, no_days_train, day
        )

        # test
        x_test_scaled_filtered, y_test_scaled_filtered, lengths_test_filtered = filter_window_data_by_length(
            X_test_scaled, y_test_scaled, valid_length_test, no_days_test, day
        )

        train_dataset_filtered = VIDataset(x_train_scaled_filtered, y_train_scaled_filtered, lengths_train_filtered)
        test_dataset_filtered = VIDataset(x_test_scaled_filtered, y_test_scaled_filtered, lengths_test_filtered)

        train_loader_filtered = DataLoader(train_dataset_filtered, batch_size=best_gru_params['batch_size'],
                                           shuffle=True)
        test_loader_filtered = DataLoader(test_dataset_filtered, batch_size=best_gru_params['batch_size'],
                                          shuffle=False)

        # Training metrics
        r2_train_filtered, mse_train_filtered, mae_train_filtered = evaluate_model(best_model, train_loader_filtered,
                                                                                   output_variable_scaler=scaler_y,
                                                                                   do_inverse_transform=True,
                                                                                   plot_pred_vs_true=False)
        # Test metrics
        r2_test_filtered, mse_test_filtered, mae_test_filtered = evaluate_model(best_model, test_loader_filtered,
                                                                                output_variable_scaler=scaler_y,
                                                                                do_inverse_transform=True,
                                                                                plot_pred_vs_true=False)

        # print(
        #     f'Train R²: {r2_train_filtered:.4f}, Train MSE: {mse_train_filtered:.4f}, Train MAE: {mae_train_filtered:.4f}')
        # print(f'Test R²: {r2_test_filtered:.4f}, Test MSE: {mse_test_filtered:.4f}, Test MAE: {mae_test_filtered:.4f}')
        all_test_r2.append(r2_test_filtered)
        all_test_mse.append(mse_test_filtered)
        all_test_mae.append(mae_test_filtered)

    return all_test_r2, all_test_mse, all_test_mae


In [ ]:
import optuna
from sklearn.metrics import r2_score

epochs = 1000

def objective2(trial):

    hidden_size = trial.suggest_categorical('hidden_size', [8, 16])
    num_layers = trial.suggest_categorical('num_layers', [1, 2, 3])
    dropout = trial.suggest_float('dropout', 0.0,  0.5)
    lr = trial.suggest_float('lr', 1e-4,  1e-2, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-10, 1e-3, log=True)
    # bidirectional = trial.suggest_categorical('bidirectional', [True, False])
    batch_size = trial.suggest_categorical('batch_size', [8, 16, 32, 64])

    mean_r2 = validation_with_test(
        trial_id = trial.number,
        rand_seed = seed,
        batch_size= batch_size,
        epochs=epochs,
        hidden_size=hidden_size,
        num_layers=num_layers,
        dropout=dropout,
        bidirectional=False,
        learning_rate=lr,
        weight_decay=weight_decay,
        hyper_param_criterion_method="L1Loss")
    return mean_r2

In [ ]:
sampler = optuna.samplers.TPESampler(seed=seed)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective2, n_trials=180)

print(f"Best R²: {study.best_value:.4f}")
print("Best hyperparameters:", study.best_params)

In [ ]:
# =============================================
#  Train the best model with best hyper params
# =============================================
from model_definitions import GRUModel


# window size = 30
# Dataset: v3_interpolated , seed = 23
# Test R²: 0.5650, Test MSE: 0.0328, Test MAE: 0.1409 max R²: 0.9164
# Features: Only VIs
# Masking/Length in GRU: False;  Make sure GRU model does not use lengths in the forward function
# Pad = 0
# criterion_method = L1 Loss
# best_gru_params = {'hidden_size': 8, 'num_layers': 3, 'dropout': 0.36877045653176815, 'lr': 0.0015060644866586401, 'weight_decay': 1.4534255491581203e-07, 'batch_size': 32}

# window size = 30
# Dataset: v3_interpolated , seed = 46
# Test R²: 0.56, Test MSE: , Test MAE:  max R²: 0.84
# Features: Only VIs
# Masking/Length in GRU: False;  Make sure GRU model does not use lengths in the forward function
# Changed the loss function to calculate based on valid days of the window
# Pad = 0
# criterion_method = L1 Loss
# best_gru_params = {'hidden_size': 8, 'num_layers': 3, 'dropout': 0.28917079428946046, 'lr': 0.00011659340413762036, 'weight_decay': 7.929486820874147e-06, 'batch_size': 64}

# window size = 30
# Dataset: v3_interpolated , seed = 23, and reproducible
# Test R²: 0.5219, Test MSE: 0.2308, Test MAE: 0.4077, max R²: 0.7179
# Features: Only VIs
# Masking/Length in GRU: False;  Make sure GRU model does not use lengths in the forward function
# Pad = 0
# criterion_method = L1 Loss
# best_gru_params = {'hidden_size': 8, 'num_layers': 1, 'dropout': 0.12327840246560814, 'lr': 0.0007042073586044224, 'weight_decay': 4.825821170848037e-09, 'batch_size': 8}


# window size = 30
# Dataset: v3_interpolated , seed = 46, and reproducible
# Test R²: 0.5804, Test MSE: 0.0316, Test MAE: 0.1434, max R²: 0.8438721299171448
# Features: Only VIs
# Masking/Length in GRU: False;  Make sure GRU model does not use lengths in the forward function
# Pad = 0
# criterion_method = L1 Loss
best_gru_params = {'hidden_size': 8, 'num_layers': 3, 'dropout': 0.02068428509102778, 'lr': 0.00013526057808875593, 'weight_decay': 2.116707558077861e-10, 'batch_size': 64}
# 5 seeds: 46, 56, 66, 76, 86
# 46: Test R²: 0.5807, Test MSE: 0.0316, Test MAE: 0.1432, Max test R²: 0.8530
# 36: Test R²: 0.2946, Test MSE: 0.0531, Test MAE: 0.1838, Max test R²: 0.6275
# 88: Test R²: 0.5057, Test MSE: 0.0372, Test MAE: 0.1587, Max test R²: 0.6961
# 56: Test R²: 0.3983, Test MSE: 0.0453, Test MAE: 0.1727, Max test R²: 0.6501
# 66: Test R²: 0.2718, Test MSE: 0.0548, Test MAE: 0.1881, Max test R²: 0.4908
# Mean of mean r2:  0.410230827331543
# Std of mean r2:  0.11905845278744281
# Mean of mean mse:  0.04440388604998589
# Std of mean mse:  0.008963944627635197
# Mean of mean mae:  0.16931569278240205
# Std of mean mae:  0.016543977762024077
# Mean of max r2:  0.663489830493927
# Std of max r2:  0.11684847635453637


# window size = 30
# Dataset: v3_interpolated , seed = 54, and reproducible
# Test R²: 0.4660, Test MSE: 0.0402, Test MAE: 0.1648, Max test R²: 0.6452
# Features: Only VIs
# Masking/Length in GRU: False;  Make sure GRU model does not use lengths in the forward function
# Pad = 0
# criterion_method = L1 Loss
# best_gru_params = {'hidden_size': 8, 'num_layers': 3, 'dropout': 0.44349554454546614, 'lr': 0.0003513911237848174, 'weight_decay': 1.5029261944853608e-07, 'batch_size': 64}
# Mean of mean r2:  0.25420795679092406
# Mean of mean mse:  0.05615089386701584
# Mean of mean mae:  0.19021715521812438
# Mean of max r2:  0.5850443243980408

# window size = 30
# Dataset: v3_interpolated , seed = 32, and reproducible
# Test R²: 0.4660, Test MSE: 0.0402, Test MAE: 0.1648, Max test R²: 0.6452
# Features: Only VIs
# Masking/Length in GRU: False;  Make sure GRU model does not use lengths in the forward function
# Pad = 0
# criterion_method = L1 Loss
# best_gru_params = {'hidden_size': 8, 'num_layers': 3, 'dropout': 0.30564146938833675, 'lr': 0.00042666715178505033, 'weight_decay': 4.558854059721666e-08, 'batch_size': 16}
# Mean of mean r2:  0.24237899780273436
# Mean of mean mse:  0.05704149901866913
# Mean of mean mae:  0.19220711588859557
# Mean of max r2:  0.5439740777015686


# window size = 30
# Dataset: v3_interpolated , seed = 48, and reproducible
# Test R²: 0.4660, Test MSE: 0.0402, Test MAE: 0.1648, Max test R²: 0.6452
# Features: Only VIs
# Masking/Length in GRU: False;  Make sure GRU model does not use lengths in the forward function
# Pad = 0
# criterion_method = L1 Loss
# best_gru_params = {'hidden_size': 8, 'num_layers': 3, 'dropout': 0.33452350649012574, 'lr': 0.007019922929218758, 'weight_decay': 5.864775094985861e-10, 'batch_size': 8}
# Mean of mean r2:  0.21870957612991332
# Mean of mean mse:  0.05882357656955719
# Mean of mean mae:  0.19672459065914155
# Mean of max r2:  0.5780048489570617


# window size = 30
# Dataset: v3_interpolated , seed = 43, and reproducible
# Test R²: 0.5784, Test MSE: 0.0317, Test MAE: 0.1508, Max test R²: 0.7736
# Features: Only VIs
# Masking/Length in GRU: False;  Make sure GRU model does not use lengths in the forward function
# Pad = 0
# criterion_method = L1 Loss
# best_gru_params = {'hidden_size': 8, 'num_layers': 2, 'dropout': 0.431689559545345, 'lr': 0.000133273341424588, 'weight_decay': 0.0006733230337449518, 'batch_size': 64}
# 58: Test R²: 0.3787, Test MSE: 0.0468, Test MAE: 0.1734, Max test R²: 0.6117
# 73: Test R²: 0.6343, Test MSE: 0.0275, Test MAE: 0.1321, Max test R²: 0.8098
# 88: Test R²: 0.2933, Test MSE: 0.0532, Test MAE: 0.1957, Max test R²: 0.4789
# 103: Test R²: 0.1479, Test MSE: 0.0642, Test MAE: 0.2066, Max test R²: 0.4191
# Mean of mean r2:  0.4065207004547119
# Std of mean r2:  0.17993556958475687
# Mean of mean mse:  0.044683223217725755
# Std of mean mse:  0.013547397534321572
# Mean of mean mae:  0.17174419164657592
# Std of mean mae:  0.027578685174668112
# Mean of max r2:  0.6186252474784851
# Std of max r2:  0.15487313949048226


# window size = 30
# Dataset: v3_interpolated , seed = 43, and reproducible
# Test R²: 0.4729, Test MSE: 0.0397, Test MAE: 0.1606, Max test R²: 0.7846
# Features: VIs + Canopy Temperature
# Masking/Length in GRU: False;  Make sure GRU model does not use lengths in the forward function
# Pad = 0
# criterion_method = L1 Loss
# best_gru_params =  {'hidden_size': 16, 'num_layers': 2, 'dropout': 0.37992104201588306, 'lr': 0.0005215254489193414, 'weight_decay': 0.0006959949639152564, 'batch_size': 32}

# window size = 30
# Dataset: v3_interpolated , seed = 23, and reproducible
# Test R²: 0.5458, Test MSE: 0.0342, Test MAE: 0.1422, Max test R²: 0.7220
# Features: VIs + Canopy Temperature
# Masking/Length in GRU: False;  Make sure GRU model does not use lengths in the forward function
# Pad = 0
# criterion_method = L1 Loss
# best_gru_params = {'hidden_size': 8, 'num_layers': 3, 'dropout': 0.48636850187164987, 'lr': 0.0008411415754436524, 'weight_decay': 0.0005596255005253192, 'batch_size': 32}
# 23: Test R²: 0.5458, Test MSE: 0.0342, Test MAE: 0.1422, Max test R²: 0.7220
# 53: Test R²: 0.3155, Test MSE: 0.0515, Test MAE: 0.1749, Max test R²: 0.6049
# 68: Test R²: 0.4552, Test MSE: 0.0410, Test MAE: 0.1575, Max test R²: 0.7378
# 83 Test R²: 0.1363, Test MSE: 0.0650, Test MAE: 0.1995, Max test R²: 0.5251
# 93: Test R²: 0.3004, Test MSE: 0.0527, Test MAE: 0.1684, Max test R²: 0.6761
# Mean of mean r2:  0.21727021932601928
# Mean of mean mse:  0.058931946754455566
# Mean of mean mae:  0.18242392241954802
# Mean of max r2:  0.6196243524551391


# window size = 30
# Dataset: v3_interpolated , seed = 34, and reproducible
# Features: VIs + Canopy Temperature
# Masking/Length in GRU: False;  Make sure GRU model does not use lengths in the forward function
# Pad = 0
# criterion_method = L1 Loss
# best_gru_params = {'hidden_size': 8, 'num_layers': 3, 'dropout': 0.2983942560313512, 'lr': 0.00718190240323899, 'weight_decay': 6.842240386260646e-09, 'batch_size': 32}

In [ ]:
from helper import make_progressive_windows
from vi_dataset import VIDataset
from sklearn.metrics import r2_score
from torch.utils.data import DataLoader

X_train_scaled, y_train_scaled, valid_length_train, no_days_train  = make_progressive_windows(
            dataframe=train_df_scaled,
            features=features,
            output_variable=output_variable,
            window_size=window_size,
            pad_value=pad_value)

X_test_scaled, y_test_scaled, valid_length_test, no_days_test = make_progressive_windows(
    dataframe=test_df_scaled,
    features=features,
    output_variable=output_variable,
    window_size=window_size,
    pad_value=pad_value)

In [ ]:
best_model = GRUModel(
input_size=len(features),
hidden_size=best_gru_params['hidden_size'],
num_layers=best_gru_params['num_layers'],
output_size=1,
dropout=best_gru_params['dropout'],
bidirectional=False).to(device)

In [ ]:
import csv
from plots import plot_r2, plot_mse

epochs = 1000
criterion_method="L1Loss"

all_mean_r2 = []
all_mae = []
all_mse = []
all_max_r2 = []

seeds_1 = [46, 36, 88, 56, 66]
seeds_2 = [23, 53, 68, 83, 93]
for rand_seed in seeds_1:
    # rand_seed = (seed + (i*15))
    # if i == 0:
    #     rand_seed = seed
    # else:
    #     rand_seed = random.randint(1, 200)


    print("--------------  Random Seed: ", rand_seed)

    g = reset_seed_all(rand_seed)

    train_dataset = VIDataset(X_train_scaled, y_train_scaled, valid_length_train)
    test_dataset = VIDataset(X_test_scaled, y_test_scaled, valid_length_test)

    train_loader  = DataLoader(train_dataset, batch_size=best_gru_params['batch_size'], shuffle=True, num_workers =0, worker_init_fn=seed_worker, generator=g)
    test_loader    = DataLoader(test_dataset, batch_size=best_gru_params['batch_size'], shuffle=False, num_workers =0, worker_init_fn=seed_worker, generator=g)

    best_model = GRUModel(
        input_size=len(features),
        hidden_size=best_gru_params['hidden_size'],
        num_layers=best_gru_params['num_layers'],
        output_size=1,
        dropout=best_gru_params['dropout'],
        bidirectional=False).to(device)

    train_data(
        model = best_model,
        model_train_loader= train_loader,
        num_epochs = epochs,
        learning_rate = best_gru_params['lr'],
        weight_decay = best_gru_params['weight_decay'],
        hyper_param_criterion_method = criterion_method,
        file_save_name=f"best_model_hyper_param999_ {rand_seed}.pth",
        stop_early=True
    )

    # best_model.load_state_dict(torch.load("best_gru_models/best_model_hyper_param_14.pth", map_location=device))

    # Training metrics
    r2_train, mse_train, mae_train = evaluate_model(best_model, train_loader, output_variable_scaler=scaler_y, do_inverse_transform=True, plot_pred_vs_true=False)
    # Test metrics
    r2_test, mse_test, mae_test = evaluate_model(best_model, test_loader, output_variable_scaler=scaler_y, do_inverse_transform=True, plot_pred_vs_true=False)

    test_r2, test_mse, test_mae = get_results_by_day()

    print(f'Train R²: {r2_train:.4f}, Train MSE: {mse_train:.4f}, Train MAE: {mae_train:.4f}')
    print(f'Test R²: {r2_test:.4f}, Test MSE: {mse_test:.4f}, Test MAE: {mae_test:.4f}, Max test R²: {np.max(test_r2):.4f}')

    # plot_r2(test_r2, total_n_days)
    # plot_mse(test_mse, total_n_days)

    START_DAY = 122 # May 1st
    gre_days = np.arange(START_DAY, START_DAY + total_n_days)


    with open(f'results/best_gru_model_14_seed_{rand_seed}_performance.csv', 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['Gregorian day', 'R2', 'MSE', 'MAE'])  # header
        for a, b, c, d in zip(gre_days, test_r2, test_mse, test_mae):
            writer.writerow([a, b,c,d])


    all_mean_r2.append(r2_test)
    all_mae.append(mae_test)
    all_mse.append(mse_test)
    all_max_r2.append(np.max(test_r2))



In [ ]:
    print("Mean of mean r2: ", np.mean(all_mean_r2))
    print("Std of mean r2: ", np.std(all_mean_r2))

    print("Mean of mean mse: ", np.mean(all_mse))
    print("Std of mean mse: ", np.std(all_mse))

    print("Mean of mean mae: ", np.mean(all_mae))
    print("Std of mean mae: ", np.std(all_mae))

    print("Mean of max r2: ", np.mean(all_max_r2))
    print("Std of max r2: ", np.std(all_max_r2))

In [ ]:
print(all_max_r2)

In [ ]:
# import matplotlib.pyplot as plt
#
# plt.figure(figsize=(8,5))
# plt.plot(range(1, 227+1), train_losses, marker='o')
# plt.xlabel('Epoch')
# plt.ylabel('Loss (MSE)')
# plt.title('Training Loss over Epochs')
# plt.grid(True)
# plt.show()

In [ ]:
test_df_scaled

In [ ]:
test_plot_14d = test_df_scaled[test_df_scaled["plot_id"] == "23_w"]
test_plot_14d

In [ ]:
X_test_plot_scaled, y_test_plot_scaled, valid_length_test_plot, no_days_test_plot = make_progressive_windows(
    dataframe=test_plot_14d,
    features=features,
    output_variable=output_variable,
    window_size=window_size,
    pad_value=pad_value)

In [ ]:
total_n_days = 46 # (from may 1st to June 15th)

y_pred_list = []
y_true_list = []

for filter_day in range(1, total_n_days + 1):
    print(f"day: {filter_day}")

    X_test_plot_scaled_filtered, y_test_plot_scaled_filtered, lengths_test_plot_filtered = filter_window_data_by_length(
        X_test_plot_scaled, y_test_plot_scaled, valid_length_test_plot, no_days_test_plot, filter_day
    )

    test_plot_dataset_filtered = VIDataset(X_test_plot_scaled_filtered, y_test_plot_scaled_filtered, lengths_test_plot_filtered)
    test_plot_loader_filtered    = DataLoader(test_plot_dataset_filtered, batch_size=best_gru_params['batch_size'], shuffle=False)

    # Test metrics
    y_true, y_predicted = predict_yield(best_model, test_plot_loader_filtered, output_variable_scaler=scaler_y, do_inverse_transform = True,  plot_pred_vs_true=False)

    print(y_true, y_predicted)

    y_pred_list.append(float(y_predicted[0]))
    y_true_list.append(float(y_true[0]))

actual_yield = y_true_list[0]

In [ ]:
from plots import plot_predicted_yield

#146 is the heading date for plot 14_d
#135 is the heading date for plot 23_w
plot_predicted_yield(total_n_days, y_pred_list, actual_yield, 135, "23_w")


In [ ]:
# save model
torch.save(best_model, "best_gru_models/best_model_full_12.pth")

# save Scalers for future predictions
import joblib

joblib.dump(scaler_X, "best_gru_models/scalers/scaler_X_12.save")
joblib.dump(scaler_y, "best_gru_models/scalers/scaler_y_12.save")